# FAKTA — Evidence Retrieval Test
Test the hybrid BM25 + embedding retrieval system.

In [ ]:
import sys
sys.path.insert(0, 'src/evidence')
from retriever import HybridRetriever
from source_scoring import get_source_credibility, compute_recency_score

In [ ]:
# Initialize retriever
retriever = HybridRetriever(chroma_path='data/evidence/chroma_db')
print(retriever.get_stats())

In [ ]:
# Add sample evidence documents
sample_docs = [
    {
        'doc_id': 'doc1',
        'text': 'BPOM menyatakan bahwa matcha aman dikonsumsi dalam jumlah normal. Tidak ada bukti bahwa matcha menyebabkan gagal ginjal.',
        'metadata': {'source': 'BPOM', 'source_tier': 2, 'url': 'https://bpom.go.id/', 'title': 'Keamanan Matcha'},
    },
    {
        'doc_id': 'doc2',
        'text': 'BMKG mencatat gempa magnitudo 5.2 di Maluku pada 15 Januari 2025. Tidak berpotensi tsunami.',
        'metadata': {'source': 'BMKG', 'source_tier': 2, 'url': 'https://bmkg.go.id/', 'title': 'Gempa Maluku'},
    },
    {
        'doc_id': 'doc3',
        'text': 'Hoaks! Klaim bahwa vaksin mengandung chip 5G tidak benar. WHO dan Kemenkes telah membantah klaim tersebut.',
        'metadata': {'source': 'TurnBackHoax', 'source_tier': 3, 'url': 'https://turnbackhoax.id/', 'title': 'Hoaks Vaksin 5G'},
    },
]

for doc in sample_docs:
    retriever.add_document(doc['doc_id'], doc['text'], doc['metadata'])

print(f"Database stats: {retriever.get_stats()}")

In [ ]:
# Test search queries
queries = [
    'matcha gagal ginjal',
    'gempa Maluku tsunami',
    'vaksin chip 5G',
    'BBM naik harga',
]

for query in queries:
    print(f"\nQuery: '{query}'")
    results = retriever.search(query, top_k=3)
    if results:
        for r in results:
            print(f"  [{r.get('source')}] score={r.get('final_score', 0):.3f} - {r.get('text', '')[:80]}...")
    else:
        print("  No results found")

In [ ]:
# Test source credibility and recency scoring
sources = ['BPOM', 'TurnBackHoax', 'Kompas', 'Wikipedia', 'Unknown']
for s in sources:
    cred = get_source_credibility(s)
    print(f"  {s}: credibility={cred:.2f}")

dates = ['2026-06-01', '2026-03-01', '2025-12-01', '2024-01-01', '']
for d in dates:
    recency = compute_recency_score(d)
    print(f"  {d or 'None'}: recency={recency:.2f}")

In [ ]:
# Test Google Fact Check API (requires API key)
import os
from factcheck_api import GoogleFactCheckAPI

api = GoogleFactCheckAPI()
if api.api_key:
    results = api.search('matcha gagal ginjal', language='id')
    print(f"Found {len(results)} fact-checks")
    for r in results:
        print(f"  [{r['source']}] {r['text'][:80]}...")
else:
    print("No GOOGLE_FACTCHECK_API_KEY set, skipping API test")